# Session 17 Solutions: Describing Data and Probability

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
exams = pd.read_csv(DATA_URL + "exam_scores.csv")

rng = np.random.default_rng(27)

## In-Class Exercises

### In-Class Exercise 1: Describe Physics

In [2]:
p = exams["physics"]
q1, q3 = p.quantile([0.25, 0.75])
print(round(p.mean(), 1), p.median(), round(p.std(), 1), q3 - q1)
print("Skew:", round(p.skew(), 2))

51.7 51.0 11.4 15.0
Skew: -0.03


### In-Class Exercise 2: Top Ten Percent

In [3]:
cut = exams["math"].quantile(0.9)
print(cut, (exams["math"] >= cut).sum())

75.0 51


### In-Class Exercise 3: Two Dice

In [4]:
total = rng.integers(1, 7, 100_000) + rng.integers(1, 7, 100_000)
print(round((total == 7).mean(), 4), round(6 / 36, 4))

0.1664 0.1667


### In-Class Exercise 4: Exam Guessing

In [5]:
print(round(1 - stats.binom.cdf(4, 10, 0.25), 4))

0.0781


## Part A: Basic

### Exercise 1: Summary Table

In [6]:
cols = ["math", "physics", "english"]
exams[cols].agg(["mean", "median", "std", "min", "max"]).T.round(1)

,mean,median,std,min,max
math,56.7,57.0,13.3,11.0,95.0
physics,51.7,51.0,11.4,15.0,87.0
english,67.3,67.0,7.6,47.0,90.0


### Exercise 2: Coin Experiments

In [7]:
for n in [10, 100, 100_000]:
    print(n, (rng.integers(0, 2, n)).mean())
# The share wanders for small numbers of flips and settles near 0.5 for many flips: the law of large numbers.

10 0.5
100 0.48
100000 0.50057


## Part B: Intermediate

### Exercise 3: Normal Probabilities

In [8]:
mu, sd = exams["math"].mean(), exams["math"].std()
print("Model below 40:", round(stats.norm.cdf(40, mu, sd), 3), "real:", round((exams["math"] < 40).mean(), 3))
print("Model above 80:", round(1 - stats.norm.cdf(80, mu, sd), 3), "real:", round((exams["math"] > 80).mean(), 3))

Model below 40: 0.104 real: 0.096
Model above 80: 0.04 real: 0.038


### Exercise 4: Churn by Two Factors

In [9]:
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = churn["churn"] == "Yes"
pt = pd.pivot_table(churn, index="contract", columns="internet", values="left", aggfunc="mean").round(3)
print(pt)
print("Overall:", round(churn["left"].mean(), 3))

internet          DSL  Fiber
contract                    
Month-to-month  0.382  0.563
One year        0.143  0.206
Two year        0.044  0.115
Overall: 0.303


## Part C: Challenge

### Exercise 5: A Queue Simulation

When customers arrive almost as fast as they are served, queues grow quickly. Simulation shows this without any queueing formula.

In [10]:
def simulate(mean_gap, customers=200):
    arrival, free_at, waits = 0.0, 0.0, []
    for _ in range(customers):
        arrival += rng.exponential(mean_gap)
        start = max(arrival, free_at)
        waits.append(start - arrival)
        free_at = start + rng.uniform(2, 4)
    return np.array(waits)

for gap in [3.0, 2.5]:
    w = simulate(gap)
    print(gap, round(w.mean(), 2), round((w > 5).mean(), 3))

3.0 16.98 0.82
2.5 33.46 0.965


## Part D: Cumulative Project: Exam Scores Study

### Exercise 6: Exam Scores Study: Part 1

In [11]:
num = exams.select_dtypes("number").drop(columns="student_id")
table = pd.DataFrame({"mean": num.mean(), "median": num.median(), "std": num.std(),
                      "IQR": num.quantile(0.75) - num.quantile(0.25)}).round(1)
print(table)
for col in ["math", "physics", "english"]:
    s = exams[col]
    within = [round(s.between(s.mean() - k * s.std(), s.mean() + k * s.std()).mean(), 3) for k in (1, 2, 3)]
    print(col, "skew", round(s.skew(), 2), "within 1, 2, 3 std:", within)
# All three subjects are close to normal: small skew and shares close to 68, 95 and 99.7 percent.
for col in ["math", "physics", "english"]:
    exams[col + "_z"] = (exams[col] - exams[col].mean()) / exams[col].std()
exams["overall_z"] = exams[["math_z", "physics_z", "english_z"]].mean(axis=1)
exams.nlargest(5, "overall_z")[["student_id", "study_hours_week", "sleep_hours", "attendance_pct", "overall_z"]].round(2)

                  mean  median   std   IQR
study_hours_week   6.9     6.0   4.0   5.4
sleep_hours        7.1     7.1   1.1   1.4
attendance_pct    87.7    88.0   7.9  12.0
math              56.7    57.0  13.3  17.0
physics           51.7    51.0  11.4  15.0
english           67.3    67.0   7.6  10.0
math skew -0.03 within 1, 2, 3 std: [np.float64(0.682), np.float64(0.952), np.float64(0.998)]
physics skew -0.03 within 1, 2, 3 std: [np.float64(0.682), np.float64(0.956), np.float64(0.996)]
english skew 0.15 within 1, 2, 3 std: [np.float64(0.678), np.float64(0.948), np.float64(1.0)]


,student_id,study_hours_week,sleep_hours,attendance_pct,overall_z
481,482,22.7,7.1,96,2.53
393,394,18.7,8.4,100,2.45
195,196,22.2,8.7,93,2.27
263,264,19.7,7.5,93,2.15
457,458,9.9,6.0,89,1.64


### Exercise 7: Exam Scores Study: Part 2

In [12]:
exams = pd.read_csv(DATA_URL + "exam_scores.csv")
print("P(passed):", round(exams["passed"].mean(), 3))
print("P(passed | study > 10):", round(exams.loc[exams["study_hours_week"] > 10, "passed"].mean(), 3))
print("P(math > 70 | sleep >= 8):", round((exams.loc[exams["sleep_hours"] >= 8, "math"] > 70).mean(), 3))
print("P(math > 70 | sleep < 6):", round((exams.loc[exams["sleep_hours"] < 6, "math"] > 70).mean(), 3))
passed = exams["passed"].to_numpy()
classes = rng.choice(passed, size=(10_000, 30))
print("P(fewer than 10 of 30 pass):", round((classes.sum(axis=1) < 10).mean(), 3))

P(passed): 0.45
P(passed | study > 10): 0.838
P(math > 70 | sleep >= 8): 0.234
P(math > 70 | sleep < 6): 0.103
P(fewer than 10 of 30 pass): 0.073


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | The median depends only on the middle of the sorted values. |
| 2 | C | That is the middle part of the 68-95-99.7 rule. |
| 3 | B | Complement rule: 1 - 0.3. |
| 4 | B | For independent events, multiply: 0.5 x 0.2. |
| 5 | B | Averages over many trials settle near the true value. |